In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import plotly.graph_objects as go
import plotly.subplots as sp
from pathlib import Path
import sys
import yfinance as yf

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

PROCESSED_DATA_DIR = project_root / "data" / "processed"
FIGURES_DIR        = project_root / "figures"
PCA_DIR            = project_root / "data" / "pca"

LOG_MONEYNESS_GRID = np.linspace(np.log(0.80), np.log(1.20), 50)
TTM_GRID           = np.linspace(14/365, 1.0, 50)
GRID_SHAPE         = (50, 50)
M, T               = np.meshgrid(LOG_MONEYNESS_GRID, TTM_GRID)

print("Config loaded.")

Config loaded.


In [4]:
# Load PC scores
scores_df = pd.read_csv(PCA_DIR / "pc_scores.csv", index_col="date", parse_dates=True)

# Load explained variance
var_df = pd.read_csv(PCA_DIR / "explained_variance.csv")

# Load mean surface
mean_df = pd.read_csv(PCA_DIR / "mean_surface.csv", index_col=0)
mean_surface = mean_df.values

# Load all processed IV files to rebuild surface matrix
processed_files = sorted(PROCESSED_DATA_DIR.glob("SPY_IV_*.csv"))
print(f"Loaded {len(processed_files)} processed days")
print(f"PC scores shape: {scores_df.shape}")
print(f"Mean surface shape: {mean_surface.shape}")

Loaded 199 processed days
PC scores shape: (199, 10)
Mean surface shape: (50, 50)


In [5]:
def nadaraya_watson(m_grid, t_grid, m_data, t_data, iv_data, h_m=0.05, h_t=0.1):
    surface = np.zeros((len(t_grid), len(m_grid)))
    for i, t in enumerate(t_grid):
        for j, m in enumerate(m_grid):
            w = (np.exp(-0.5 * ((m_data - m) / h_m)**2) *
                 np.exp(-0.5 * ((t_data - t) / h_t)**2))
            if w.sum() > 0:
                surface[i, j] = np.dot(w, iv_data) / w.sum()
            else:
                surface[i, j] = np.nan
    return surface

# Rebuild surface matrix
surface_matrix = []
dates = []
for f in processed_files:
    date = f.stem.replace("SPY_IV_", "")
    df = pd.read_csv(f)
    surface = nadaraya_watson(
        LOG_MONEYNESS_GRID, TTM_GRID,
        df["log_moneyness"].values,
        df["TTM"].values,
        df["IV"].values
    )
    nan_pct = np.isnan(surface).mean()
    if nan_pct > 0.1:
        continue
    surface = np.where(np.isnan(surface), np.nanmean(surface), surface)
    surface_matrix.append(surface.flatten())
    dates.append(date)

surface_matrix = np.array(surface_matrix)
dates_dt = pd.to_datetime(dates)
print(f"Surface matrix: {surface_matrix.shape}")

Surface matrix: (199, 2500)


In [6]:
fig = go.Figure(data=[go.Surface(
    x=LOG_MONEYNESS_GRID,
    y=TTM_GRID,
    z=mean_surface,
    colorscale='Viridis',
    colorbar=dict(title='IV')
)])

fig.update_layout(
    title="Mean SPY IV Surface — 2025",
    scene=dict(
        xaxis_title="Log-Moneyness (log(K/S))",
        yaxis_title="Time to Maturity (years)",
        zaxis_title="Implied Volatility",
        camera=dict(eye=dict(x=1.5, y=-1.5, z=0.8))
    ),
    width=900, height=650
)
fig.show()

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

explained = var_df["explained_variance"].values
cumulative = var_df["cumulative_variance"].values

axes[0].bar(range(1, 11), explained[:10] * 100, color='steelblue', alpha=0.8)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance (%)")
axes[0].set_title("Scree Plot — SPY IV Surface 2025")
axes[0].set_xticks(range(1, 11))

axes[1].plot(range(1, 11), cumulative[:10] * 100,
             marker='o', color='steelblue', linewidth=2)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90%')
axes[1].axhline(y=95, color='orange', linestyle='--', alpha=0.5, label='95%')
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance (%)")
axes[1].set_title("Cumulative Explained Variance")
axes[1].set_xticks(range(1, 11))
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "summary_scree_plot.png", dpi=150)
plt.show()
print("Scree plot saved.")

Scree plot saved.


/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_62508/589092565.py:24: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



In [8]:
from sklearn.decomposition import PCA

mean_surface_vec = surface_matrix.mean(axis=0)
X = surface_matrix - mean_surface_vec
pca = PCA()
pca.fit(X)

fig = sp.make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "surface"}, {"type": "surface"}, {"type": "surface"}]],
    subplot_titles=[
        f"PC1 — {pca.explained_variance_ratio_[0]:.1%} variance",
        f"PC2 — {pca.explained_variance_ratio_[1]:.1%} variance",
        f"PC3 — {pca.explained_variance_ratio_[2]:.1%} variance"
    ]
)

for i in range(3):
    loading = pca.components_[i].reshape(GRID_SHAPE)
    fig.add_trace(
        go.Surface(
            x=LOG_MONEYNESS_GRID,
            y=TTM_GRID,
            z=loading,
            colorscale='RdBu',
            showscale=(i == 0),
            colorbar=dict(title='Loading', x=1.02) if i == 0 else None
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title="Principal Component Loadings — SPY IV Surface 2025",
    width=1200, height=500
)
fig.show()

In [9]:
fig = sp.make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f"PC1 Score ({pca.explained_variance_ratio_[0]:.1%} variance) — Level",
        f"PC2 Score ({pca.explained_variance_ratio_[1]:.1%} variance) — Slope",
        f"PC3 Score ({pca.explained_variance_ratio_[2]:.1%} variance) — Curvature"
    ],
    vertical_spacing=0.08
)

colors = ['steelblue', 'darkorange', 'green']
scores = pca.transform(X)

for i in range(3):
    fig.add_trace(
        go.Scatter(
            x=dates_dt,
            y=scores[:, i],
            mode='lines',
            line=dict(color=colors[i], width=1.5),
            name=f"PC{i+1}",
            showlegend=False
        ),
        row=i+1, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray",
                  line_width=0.8, row=i+1, col=1)

fig.update_layout(
    title="PC Score Time Series — SPY IV Surface 2025",
    width=1000, height=700,
    xaxis3_title="Date"
)
fig.show()

In [10]:
# Fetch VIX for 2025
import yfinance as yf
vix = yf.download("^VIX", start="2025-01-01", end="2025-12-31", progress=False)
vix.index = vix.index.strftime("%Y-%m-%d")

# Align with our dates
common_dates = [d for d in dates if d in vix.index]
vix_values = vix.loc[common_dates, "Close"].values
pc1_values = scores[[dates.index(d) for d in common_dates], 0]
pc2_values = scores[[dates.index(d) for d in common_dates], 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs VIX
axes[0].scatter(vix_values, pc1_values, alpha=0.5, color='steelblue', s=20)
axes[0].set_xlabel("VIX")
axes[0].set_ylabel("PC1 Score")
axes[0].set_title("PC1 Score vs VIX")
corr1 = np.corrcoef(vix_values, pc1_values)[0, 1]
axes[0].text(0.05, 0.95, f"r = {corr1:.3f}",
             transform=axes[0].transAxes, fontsize=12,
             verticalalignment='top')

# PC2 vs VIX
axes[1].scatter(vix_values, pc2_values, alpha=0.5, color='darkorange', s=20)
axes[1].set_xlabel("VIX")
axes[1].set_ylabel("PC2 Score")
axes[1].set_title("PC2 Score vs VIX")
corr2 = np.corrcoef(vix_values, pc2_values)[0, 1]
axes[1].text(0.05, 0.95, f"r = {corr2:.3f}",
             transform=axes[1].transAxes, fontsize=12,
             verticalalignment='top')

plt.suptitle("PC Scores vs VIX — SPY IV Surface 2025")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "summary_pc_vs_vix.png", dpi=150)
plt.show()
print(f"PC1 vs VIX correlation: {corr1:.3f}")
print(f"PC2 vs VIX correlation: {corr2:.3f}")

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 1 and the array at index 1 has size 199

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'darkorange', 'green']
labels = ['Level', 'Slope', 'Curvature']

for i in range(3):
    axes[i].hist(scores[:, i], bins=30, color=colors[i],
                 alpha=0.75, edgecolor='white')
    axes[i].axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    axes[i].set_xlabel(f"PC{i+1} Score")
    axes[i].set_ylabel("Frequency")
    axes[i].set_title(f"PC{i+1} ({labels[i]}) — "
                      f"μ={scores[:,i].mean():.2f}, "
                      f"σ={scores[:,i].std():.2f}")

plt.suptitle("Distribution of PC Scores — SPY IV Surface 2025")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "summary_pc_distributions.png", dpi=150)
plt.show()
print("PC distributions saved.")

PC distributions saved.


/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_52464/1051627319.py:18: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

z_min = np.nanmin(surface_matrix) * 0.9
z_max = np.nanmax(surface_matrix) * 1.1

def update(frame):
    ax.cla()
    surface = surface_matrix[frame].reshape(GRID_SHAPE)
    ax.plot_surface(M, T, surface, cmap='viridis', alpha=0.8)
    ax.set_xlabel("Log-Moneyness")
    ax.set_ylabel("TTM (years)")
    ax.set_zlabel("Implied Volatility")
    ax.set_title(f"SPY IV Surface — {dates[frame]}")
    ax.set_zlim(z_min, z_max)

anim = FuncAnimation(fig, update, frames=len(dates), interval=100)
anim.save(FIGURES_DIR / "summary_iv_animation.gif",
          writer='pillow', fps=10, dpi=100)
plt.close(fig)
print(f"Animation saved — {len(dates)} frames.")

Animation saved — 199 frames.


In [ ]:
print("=" * 50)
print("SPY IMPLIED VOLATILITY SURFACE PCA — 2025")
print("=" * 50)
print(f"\nData: {len(dates)} trading days")
print(f"Date range: {dates[0]} to {dates[-1]}")
print(f"\nPCA Results:")
print(f"  PC1 (Level):     {pca.explained_variance_ratio_[0]:.1%} variance explained")
print(f"  PC2 (Slope):     {pca.explained_variance_ratio_[1]:.1%} variance explained")
print(f"  PC3 (Curvature): {pca.explained_variance_ratio_[2]:.1%} variance explained")
print(f"  Top 3 combined:  {pca.explained_variance_ratio_[:3].sum():.1%} variance explained")
print(f"\nPC1 vs VIX correlation: {corr1:.3f}")
print(f"PC2 vs VIX correlation: {corr2:.3f}")
print(f"\nFigures saved to: {FIGURES_DIR}")
print("=" * 50)